# 🔍 Local RAG Pipeline

A local Retrieval-Augmented Generation (RAG) pipeline using:
- **ChromaDB** for vector storage
- **SentenceTransformers** (`all-MiniLM-L6-v2`) for embeddings
- **Groq / LLaMA 3** as the language model
- **BM25** for keyword-based pre-filtering (HyDE technique)
- **pdfplumber** for PDF text extraction

## 1. Import Packages

In [13]:
import os
import re

import pandas as pd
import requests
from pprint import pprint
from dotenv import dotenv_values

from groq import Groq
from google import genai
from google.genai import types
from openai import OpenAI

import chromadb
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer

import pdfplumber
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

## 2. Configuration

Load API keys and secrets from the `.env` file.

In [14]:
# Load environment variables
config = dotenv_values(".env")

GROQ_API_KEY  = config["GROQ_API_KEY"]
GEMINI_API_KEY = config["GEMINI_API_KEY"]
HF_TOKEN      = config.get("HF_TOKEN")

# Make HuggingFace token available to the transformers library
os.environ["HF_TOKEN"] = HF_TOKEN

## 3. Embedding Model

Load `all-MiniLM-L6-v2` from SentenceTransformers. This model converts text into 384-dimensional vectors.

In [15]:
model = SentenceTransformer("all-MiniLM-L6-v2")

# Quick sanity check
sample_sentence = "What is the Meaning of Life?"
sample_embedding = model.encode(sample_sentence)
print(f"Embedding shape: {sample_embedding.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2734.23it/s]


Embedding shape: (384,)


## 4. Vector Database (ChromaDB)

Initialize a persistent ChromaDB collection. Documents will be stored on disk under `./course/`.

In [16]:
chroma_client    = chromadb.PersistentClient(path="course")
collection_name  = "course"
embedding_fn     = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = chroma_client.get_or_create_collection(
    name=collection_name,
    embedding_function=embedding_fn
)
print(collection)

Collection(name=course)


## 5. PDF Text Extraction

Read a PDF file page by page, clean the extracted text, and return a list of page documents.

In [17]:
def get_text_from_pdf(path):
    documents = []
    with pdfplumber.open(path) as pdf:
        print(f"Number of pages: {len(pdf.pages)}")

        for i, page in enumerate(pdf.pages):
            text = page.extract_text()

            if text:
                # Remove leading/trailing whitespace
                text = text.strip()

                # Fix soft line breaks inside paragraphs
                text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

                # Normalize multiple spaces / newlines
                text = re.sub(r'\s+', ' ', text)

                documents.append({"id": i, "text": text})
            else:
                documents.append("")

    return documents


documents = get_text_from_pdf("xml_course.pdf")
print(f"Total pages loaded: {len(documents)}")

Number of pages: 55
Total pages loaded: 55


## 6. Text Chunking

Split each page's text into overlapping fixed-size chunks so that long pages fit within embedding/LLM context limits.

In [18]:
def split_text(text, chunk_size=1000, chunk_overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks

In [19]:
# Preview the first document
documents[0]

{'id': 0,
 'text': 'XML : Cours et exercices corrigés Master Web Intelligence et Sciences des Données (WISD) Prof. El Habib NFAOUI (elhabib.nfaoui@usmba.ac.ma) Edition 2020'}

In [20]:
# Split all documents into chunks
chunked_documents = []
for doc in documents:
    chunks = split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            "id"  : f"{doc['id']}_chunk{i + 1}",
            "text": chunk
        })

print(f"Total chunks: {len(chunked_documents)}")

Total chunks: 93


## 7. Embedding Generation

Encode each text chunk into a dense vector using the SentenceTransformer model.

In [21]:
def get_embedding(text):
    """Return the embedding vector for a given text string."""
    embeddings = model.encode(text)
    return embeddings

In [22]:
# Generate and attach embeddings to every chunk
for doc_chunk in chunked_documents:
    doc_chunk["embeddings"] = get_embedding(doc_chunk["text"])

## 8. Index Chunks into ChromaDB

Upsert all chunks (text + embedding) into the persistent vector store.

In [23]:
for doc_chunk in chunked_documents:
    collection.upsert(
        ids       =[doc_chunk["id"]],
        documents =[doc_chunk["text"]],
        embeddings=[doc_chunk["embeddings"]]
    )

print("Indexing complete.")

Indexing complete.


## 9. LLM Integration (Groq / LLaMA 3)

Use the Groq API with `llama-3.1-8b-instant` to generate a hypothetical document for a user question (HyDE — Hypothetical Document Embeddings).

In [29]:
client_groq = Groq(api_key=GROQ_API_KEY)

def get_llm_documents(question):
    """Generate a short hypothetical documentation passage for `question`."""
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role"   : "system",
                "content": (
                    f"You are a technical documentation writer. "
                    f"Write one clear, structured documentation for: {question}. "
                    f"Use simple English words, be brief, and stay on topic."
                ),
            }
        ],
        temperature=1,
        max_completion_tokens=1024,
        top_p=1,
        stream=True,
        stop=None
    )

    res = [chunk.choices[0].delta.content for chunk in completion]
    res = [s for s in res if s]
    return "".join(res)

In [30]:
# Test HyDE document generation
response = get_llm_documents(question="What is AI?")
print(response)

**What is AI?**

**Introduction**

Artificial Intelligence (AI) is a technology that enables machines to think and act like humans. It is used in various applications, from simple chatbots to complex systems like self-driving cars.

**Key Features of AI**

1. **Machine Learning**: AI allows machines to learn from experience and improve their performance over time.
2. **Reasoning**: AI can analyze data, identify patterns, and make decisions based on that analysis.
3. **Problem-Solving**: AI can solve complex problems and optimize systems.

**Components of AI**

1. **Algorithms**: These are the instructions that guide AI systems in making decisions and taking actions.
2. **Data**: AI requires large amounts of data to learn and improve.
3. **Hardware**: AI systems require powerful computers and specialized hardware to process large amounts of data.

**Types of AI**

1. **Narrow or Weak AI**: Designed to perform a specific task, such as facial recognition or language translation.
2. **Gene

## 10. LLM Document Splitting & Embedding

Break the hypothetical LLM response into paragraph-level chunks and embed them.

In [32]:
def split_text_llm(text):
    """Split LLM-generated text by newlines, keeping only substantive paragraphs."""
    chunks = []
    for i, paragraph in enumerate(text.split("\n")):
        paragraph = paragraph.strip()
        if len(paragraph) > 50:
            chunks.append({
                "id"       : f"chunk_{i}",
                "text"     : paragraph,
                "embedding": None
            })
    return chunks

In [33]:
def get_llm_embedding(response):
    """Split and embed an LLM-generated response."""
    llm_documents = split_text_llm(response)
    for doc_chunk in llm_documents:
        doc_chunk["embeddings"] = get_embedding(doc_chunk["text"])
    return llm_documents

## 11. BM25 Keyword Search

`collection.query()` does semantic (dense) search. Here we add a BM25 (sparse) keyword pass to pre-filter candidates before re-ranking by cosine similarity.

In [34]:
def bm25_search(bm25, question, chunked_documents, top_k=2):
    """
    Return the top-k chunks most relevant to `question` using BM25 scoring.

    Parameters
    ----------
    bm25             : BM25Okapi  — pre-built BM25 index
    question         : str        — user query
    chunked_documents: list[dict] — original chunk list (text + embeddings)
    top_k            : int        — number of results to return
    """
    tokenized_query = question.split()
    scores          = bm25.get_scores(tokenized_query)
    top_indices     = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

    return [
        {
            "text"      : chunked_documents[i]["text"],
            "embeddings": chunked_documents[i]["embeddings"],
            "score"     : scores[i]
        }
        for i in top_indices
    ]

## 12. Full Retrieval Pipeline (HyDE + BM25)

Combine HyDE (Hypothetical Document Embeddings) with BM25 keyword filtering:

1. Generate a hypothetical answer with the LLM  
2. Embed the hypothetical answer  
3. Use BM25 to filter the most keyword-relevant corpus chunks

In [35]:
question = "utilisé pour une partie du cours intitulé XML ?"

# Step 1 — Generate a hypothetical answer (HyDE)
hyde_doc = get_llm_documents(question)

# Step 2 — Embed the hypothetical answer
hyde_doc_embedding = get_llm_embedding(hyde_doc)
print("HyDE embeddings:", hyde_doc_embedding)

# Step 3 — BM25 pre-filtering on the corpus
corpus = [doc["text"].split() for doc in chunked_documents]
bm25   = BM25Okapi(corpus)
filtered_chunks = bm25_search(bm25, question=question, chunked_documents=chunked_documents)
print("\nBM25 top chunks:", filtered_chunks)

HyDE embeddings: [{'id': 'chunk_0', 'text': '**Documentations XML pour le cours "Intégration de données et Web"**', 'embedding': None, 'embeddings': array([-7.20040724e-02,  7.62504116e-02, -1.16539691e-02, -1.04885347e-01,
       -1.74246598e-02, -5.74563853e-02,  1.87204853e-02,  1.08024709e-01,
        8.11634492e-03, -2.20239051e-02,  5.41425385e-02, -6.05261810e-02,
        6.33426979e-02, -8.73399973e-02, -4.97222180e-03,  2.84574144e-02,
       -3.36461328e-02,  1.80318803e-02,  5.46718799e-02, -4.25896049e-03,
        7.20388293e-02,  4.44500446e-02, -2.64388546e-02, -6.63029402e-03,
        4.02482878e-03,  5.22293970e-02, -6.89016050e-03, -1.11434993e-03,
        3.77191231e-02, -4.20294981e-03, -2.37856116e-02,  1.25308428e-02,
       -1.12661365e-02,  3.57827097e-02, -2.16386393e-02, -4.14257161e-02,
        7.57461488e-02, -1.25053197e-01, -1.60512477e-02,  7.83813894e-02,
       -6.27068132e-02,  3.81032303e-02, -5.11070080e-02, -5.99228255e-02,
       -4.80893441e-02,  6

In [37]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def rerank_with_hyde(hyde_embeddings, bm25_chunks, top_k=2):
    """
    Compare les chunks réels filtrés par BM25
    avec les embeddings hypothétiques HyDE.
    Retourne les chunks classés par similarité cosine.
    """
    # Extraire les vecteurs HyDE  (shape: [n_hyde, 384])
    hyde_vecs = np.array([doc["embeddings"] for doc in hyde_embeddings])

    results = []
    for chunk in bm25_chunks:
        chunk_vec = np.array(chunk["embeddings"]).reshape(1, -1)

        # Cosine similarity entre ce chunk et chaque vecteur HyDE
        scores = cosine_similarity(chunk_vec, hyde_vecs)  # shape: [1, n_hyde]
        best_score = float(scores.max())                  # prendre le max

        results.append({
            "text" : chunk["text"],
            "score": best_score
        })

    # Trier du plus similaire au moins
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]


# ─── Exemple simple ───────────────────────────────────────────────────────────

question    = "What is XML used for?"
hyde_doc    = get_llm_documents(question)          # réponse hypothétique
hyde_embeds = get_llm_embedding(hyde_doc)          # embeddings HyDE

corpus      = [doc["text"].split() for doc in chunked_documents]
bm25        = BM25Okapi(corpus)
bm25_chunks = bm25_search(bm25, question, chunked_documents, top_k=5)

# ④ Re-ranking cosine
top_chunks = rerank_with_hyde(hyde_embeds, bm25_chunks, top_k=2)

for i, chunk in enumerate(top_chunks):
    print(f"[{i+1}] score={chunk['score']:.3f}")
    print(f"     {chunk['text']}...")
    print()

[1] score=0.641
     Chapitre 1 Structure des documents XML 1. Introduction XML est un méta-langage universel pour représenter les données échangées sur le Web qui permet au développeur de délivrer du contenu depuis les applications à d'autres applications ou aux navigateurs. XML fournit simplement une structure fondamentale et un ensemble de règles qui doivent être respectées par tous les langages de balisage. XML standardise la manière dont l'information est : - échangée - présentée - archivée - retrouvée - transformée - ... 2. Structure d’un document XML Un document XML est un document texte lisible. Un document XML est découpé en éléments structurés hiérarchiquement. Un document a un élément racine appelé élément du document.  Un élément est composé : ◼ d’un nom qui spécifie son type, ◼ d’attributs, ◼ d’un contenu formé d’éléments ou de textes. <producteur> <adresse> <rue>Azilal</rue> <ville>Casa</ville> 5...

[2] score=0.545
     Chapitre 3 XML Schema 1. Insuffisance des DTD Les 